# Proyecto 5: Segmentación de Clientes mediante Clustering
## Notebook 4: K-means Clustering

### Objetivo
Implementar el algoritmo K-means para segmentar clientes mayoristas en grupos homogéneos con características similares.

### K-means: Algoritmo
**Definición:** Particiona los datos en k clusters minimizando la inercia (suma de distancias al cuadrado desde cada punto a su centroide).

**Proceso:**
1. Inicializar k centroides aleatoriamente
2. Asignar cada punto al centroide más cercano
3. Recalcular los centroides como el promedio de puntos en cada cluster
4. Repetir 2-3 hasta convergencia

**Ventajas:** Rápido, escalable, interpretable
**Desventajas:** Sensible a inicialización, asume clusters esféricos

## 1. Importación y Preparación de Datos

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
from sklearn.metrics import silhouette_score, davies_bouldin_score, calinski_harabasz_score
from mpl_toolkits.mplot3d import Axes3D
import warnings
warnings.filterwarnings('ignore')

# Configuración visual
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (14, 8)
plt.rcParams['font.size'] = 10

print("✓ Librerías importadas")

In [ ]:
# Cargar y preparar datos
ruta_datos = '../data/Wholesale customers data.csv'
df_original = pd.read_csv(ruta_datos)

# Estandarizar
variables_numericas = ['Fresh', 'Milk', 'Grocery', 'Frozen', 'Detergents_Paper', 'Delicassen']
scaler = StandardScaler()
df_estandarizado = df_original.copy()
df_estandarizado[variables_numericas] = scaler.fit_transform(df_original[variables_numericas])

X = df_estandarizado[variables_numericas].values

print(f"✓ Datos preparados: {X.shape}")

## 2. K-means con k=3

In [ ]:
# Aplicar K-means con k=3
k_optimo = 3
kmeans_3 = KMeans(n_clusters=k_optimo, random_state=42, n_init=10)
labels_3 = kmeans_3.fit_predict(X)

# Agregar labels al dataframe
df_estandarizado['K-means_3'] = labels_3
df_original['K-means_3'] = labels_3

print(f"✓ K-means con k={k_optimo} aplicado")
print(f"\nDistribución de clusters:")
print(df_estandarizado['K-means_3'].value_counts().sort_index())

# Calcular métricas
inertia_3 = kmeans_3.inertia_
silhouette_3 = silhouette_score(X, labels_3)
davies_bouldin_3 = davies_bouldin_score(X, labels_3)
calinski_3 = calinski_harabasz_score(X, labels_3)

print(f"\nMétricas de Calidad (k={k_optimo}):")
print(f"  • Inercia: {inertia_3:>10.2f}")
print(f"  • Silhouette Score: {silhouette_3:>10.3f} (rango: -1 a 1, mayor es mejor)")
print(f"  • Davies-Bouldin Index: {davies_bouldin_3:>10.3f} (menor es mejor)")
print(f"  • Calinski-Harabasz: {calinski_3:>10.2f} (mayor es mejor)")

## 3. Análisis de Centros de Clusters (k=3)

In [ ]:
# Obtener centros (datos estandarizados)
centros_estandarizados = kmeans_3.cluster_centers_

# Convertir centros a escala original
centros_originales = scaler.inverse_transform(centros_estandarizados)

print(f"\nCENTROS DE CLUSTERS (Escala Original)\n" + "="*80)
print(f"\n{'Cluster':<10} {'Fresh':<15} {'Milk':<15} {'Grocery':<15} {'Frozen':<15} {'Detergents':<15} {'Delicassen':<15}")
print("-" * 100)

for i in range(k_optimo):
    print(f"\nCluster {i}:")
    for j, var in enumerate(variables_numericas):
        print(f"  • {var:20s}: ${centros_originales[i, j]:>10,.0f}", end="")
        if (j + 1) % 3 == 0:
            print()

print(f"\n" + "="*80)

## 4. Visualización de Clusters en 2D (usando PCA)

In [ ]:
# Aplicar PCA para visualización
pca = PCA(n_components=2)
X_pca = pca.fit_transform(X)
centros_pca = pca.transform(centros_estandarizados)

# Crear visualización
plt.figure(figsize=(12, 8))

# Colores para clusters
colores_cluster = ['#FF6B6B', '#4ECDC4', '#45B7D1']

# Plotear puntos de datos
for i in range(k_optimo):
    mask = labels_3 == i
    plt.scatter(X_pca[mask, 0], X_pca[mask, 1], 
               c=colores_cluster[i], label=f'Cluster {i}', 
               s=50, alpha=0.7, edgecolors='black', linewidth=0.5)

# Plotear centros
plt.scatter(centros_pca[:, 0], centros_pca[:, 1], 
           c='yellow', marker='*', s=800, edgecolors='red', linewidth=2,
           label='Centroides', zorder=5)

plt.xlabel(f'PC1 ({pca.explained_variance_ratio_[0]*100:.1f}%)', fontweight='bold', fontsize=11)
plt.ylabel(f'PC2 ({pca.explained_variance_ratio_[1]*100:.1f}%)', fontweight='bold', fontsize=11)
plt.title('K-means Clustering (k=3)\nVisualización en 2D usando PCA', fontsize=13, fontweight='bold', pad=15)
plt.legend(loc='best', fontsize=11)
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print(f"\n✓ Visualización en 2D completada")
print(f"Los dos primeros componentes explican {pca.explained_variance_ratio_.sum()*100:.1f}% de la varianza")

## 5. Comparación con k=4 y k=5

In [ ]:
# Entrenar modelos con diferentes k
resultados_k = {}
modelos_k = {}

for k in [3, 4, 5]:
    kmeans = KMeans(n_clusters=k, random_state=42, n_init=10)
    labels = kmeans.fit_predict(X)
    
    resultados_k[k] = {
        'modelo': kmeans,
        'labels': labels,
        'inertia': kmeans.inertia_,
        'silhouette': silhouette_score(X, labels),
        'davies_bouldin': davies_bouldin_score(X, labels),
        'calinski': calinski_harabasz_score(X, labels)
    }
    modelos_k[k] = kmeans

print("COMPARACIÓN DE MÉTRICAS POR k\n" + "="*80)
print(f"\n{'k':<5} {'Inercia':<12} {'Silhouette':<12} {'Davies-B.':<12} {'Calinski-H.':<12}")
print("-" * 80)

for k in sorted(resultados_k.keys()):
    res = resultados_k[k]
    print(f"{k:<5} {res['inertia']:<12.0f} {res['silhouette']:<12.3f} {res['davies_bouldin']:<12.3f} {res['calinski']:<12.2f}")

In [ ]:
# Visualizar comparación de métricas
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

k_values = sorted(resultados_k.keys())
metricas = {
    'Inercia': [resultados_k[k]['inertia'] for k in k_values],
    'Silhouette Score': [resultados_k[k]['silhouette'] for k in k_values],
    'Davies-Bouldin': [resultados_k[k]['davies_bouldin'] for k in k_values],
    'Calinski-Harabasz': [resultados_k[k]['calinski'] for k in k_values]
}

# Inercia
axes[0, 0].plot(k_values, metricas['Inercia'], 'o-', linewidth=2, markersize=8, color='steelblue')
axes[0, 0].axvline(x=3, color='red', linestyle='--', alpha=0.5, label='Recomendado (k=3)')
axes[0, 0].set_xlabel('k', fontweight='bold')
axes[0, 0].set_ylabel('Inercia', fontweight='bold')
axes[0, 0].set_title('Inercia vs k', fontsize=11, fontweight='bold')
axes[0, 0].grid(alpha=0.3)
axes[0, 0].legend()

# Silhouette
axes[0, 1].plot(k_values, metricas['Silhouette Score'], 'o-', linewidth=2, markersize=8, color='green')
axes[0, 1].axvline(x=3, color='red', linestyle='--', alpha=0.5, label='Recomendado (k=3)')
axes[0, 1].set_xlabel('k', fontweight='bold')
axes[0, 1].set_ylabel('Silhouette Score', fontweight='bold')
axes[0, 1].set_title('Silhouette Score vs k (↑ mejor)', fontsize=11, fontweight='bold')
axes[0, 1].grid(alpha=0.3)
axes[0, 1].legend()

# Davies-Bouldin
axes[1, 0].plot(k_values, metricas['Davies-Bouldin'], 'o-', linewidth=2, markersize=8, color='orange')
axes[1, 0].axvline(x=3, color='red', linestyle='--', alpha=0.5, label='Recomendado (k=3)')
axes[1, 0].set_xlabel('k', fontweight='bold')
axes[1, 0].set_ylabel('Davies-Bouldin', fontweight='bold')
axes[1, 0].set_title('Davies-Bouldin Index vs k (↓ mejor)', fontsize=11, fontweight='bold')
axes[1, 0].grid(alpha=0.3)
axes[1, 0].legend()

# Calinski-Harabasz
axes[1, 1].plot(k_values, metricas['Calinski-Harabasz'], 'o-', linewidth=2, markersize=8, color='purple')
axes[1, 1].axvline(x=3, color='red', linestyle='--', alpha=0.5, label='Recomendado (k=3)')
axes[1, 1].set_xlabel('k', fontweight='bold')
axes[1, 1].set_ylabel('Calinski-Harabasz', fontweight='bold')
axes[1, 1].set_title('Calinski-Harabasz Index vs k (↑ mejor)', fontsize=11, fontweight='bold')
axes[1, 1].grid(alpha=0.3)
axes[1, 1].legend()

plt.tight_layout()
plt.show()

## 6. Análisis de Características por Cluster (k=3)

In [ ]:
# Análisis detallado de cada cluster
print("ANÁLISIS DE CARACTERÍSTICAS POR CLUSTER (k=3)\n" + "="*80)

for cluster in range(k_optimo):
    df_cluster = df_original[df_original['K-means_3'] == cluster]
    
    print(f"\n{'CLUSTER ' + str(cluster):<40} (n = {len(df_cluster)} clientes | {len(df_cluster)/len(df_original)*100:.1f}%)")
    print("-" * 80)
    
    # Gasto promedio
    print("\nGasto Promedio:\n")
    for var in variables_numericas:
        promedio = df_cluster[var].mean()
        total_prom = df_cluster[variables_numericas].sum(axis=1).mean()
        pct = (promedio / total_prom) * 100
        print(f"  • {var:20s}: ${promedio:>10,.0f} ({pct:>5.1f}% del gasto total)")
    
    # Canales presentes
    print(f"\nComposición por Canal:")
    for ch in [1, 2]:
        count = len(df_cluster[df_cluster['Channel'] == ch])
        pct = (count / len(df_cluster)) * 100
        ch_name = 'HoReCa' if ch == 1 else 'Retail'
        print(f"  • {ch_name}: {count} ({pct:.1f}%)")
    
    # Regiones presentes
    print(f"\nComposición por Región:")
    for region in sorted(df_cluster['Region'].unique()):
        count = len(df_cluster[df_cluster['Region'] == region])
        pct = (count / len(df_cluster)) * 100
        print(f"  • Región {region}: {count} ({pct:.1f}%)")

## 7. Visualización 3D de Clusters

In [ ]:
# Visualización 3D
pca_3d = PCA(n_components=3)
X_pca_3d = pca_3d.fit_transform(X)
centros_pca_3d = pca_3d.transform(centros_estandarizados)

fig = plt.figure(figsize=(14, 10))
ax = fig.add_subplot(111, projection='3d')

colores_cluster = ['#FF6B6B', '#4ECDC4', '#45B7D1']

for i in range(k_optimo):
    mask = labels_3 == i
    ax.scatter(X_pca_3d[mask, 0], X_pca_3d[mask, 1], X_pca_3d[mask, 2],
              c=colores_cluster[i], label=f'Cluster {i}', s=50, alpha=0.6, edgecolors='black', linewidth=0.5)

# Plotear centros
ax.scatter(centros_pca_3d[:, 0], centros_pca_3d[:, 1], centros_pca_3d[:, 2],
          c='yellow', marker='*', s=1000, edgecolors='red', linewidth=2, label='Centroides')

ax.set_xlabel(f'PC1 ({pca_3d.explained_variance_ratio_[0]*100:.1f}%)', fontweight='bold')
ax.set_ylabel(f'PC2 ({pca_3d.explained_variance_ratio_[1]*100:.1f}%)', fontweight='bold')
ax.set_zlabel(f'PC3 ({pca_3d.explained_variance_ratio_[2]*100:.1f}%)', fontweight='bold')
ax.set_title('K-means Clustering (k=3)\nVisualización 3D usando PCA', fontsize=13, fontweight='bold')
ax.legend(loc='best')

plt.tight_layout()
plt.show()

print(f"Los 3 primeros componentes explican {pca_3d.explained_variance_ratio_.sum()*100:.1f}% de la varianza")

## 8. Análisis de Distancias Intra-cluster

In [ ]:
from scipy.spatial.distance import cdist

# Calcular distancias de cada punto a su centroide
distances = np.min(cdist(X, centros_estandarizados), axis=1)
df_estandarizado['distancia_centroide'] = distances

print("ANÁLISIS DE COHESIÓN - DISTANCIAS AL CENTROIDE\n" + "="*80)

for cluster in range(k_optimo):
    dist_cluster = distances[labels_3 == cluster]
    print(f"\nCluster {cluster}:")
    print(f"  • Distancia media: {dist_cluster.mean():.3f}")
    print(f"  • Desv. Est.: {dist_cluster.std():.3f}")
    print(f"  • Distancia mín: {dist_cluster.min():.3f}")
    print(f"  • Distancia máx: {dist_cluster.max():.3f}")
    print(f"  • Mediana: {np.median(dist_cluster):.3f}")

# Visualizar distribuciones de distancias
fig, ax = plt.subplots(figsize=(12, 6))

for cluster in range(k_optimo):
    dist_cluster = distances[labels_3 == cluster]
    ax.hist(dist_cluster, bins=20, alpha=0.6, label=f'Cluster {cluster}', edgecolor='black')

ax.set_xlabel('Distancia al Centroide', fontweight='bold')
ax.set_ylabel('Frecuencia', fontweight='bold')
ax.set_title('Distribución de Distancias Intra-cluster', fontsize=12, fontweight='bold')
ax.legend()
ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.show()

## 9. Estabilidad del Algoritmo

In [ ]:
# Evaluar estabilidad con múltiples inicializaciones
n_init = 50
silhouette_scores_init = []
inertias_init = []

for init in range(n_init):
    kmeans_temp = KMeans(n_clusters=3, random_state=init, n_init=1)
    labels_temp = kmeans_temp.fit_predict(X)
    silhouette_scores_init.append(silhouette_score(X, labels_temp))
    inertias_init.append(kmeans_temp.inertia_)

print(f"\nESTABILIDAD DEL ALGORITMO (k=3, {n_init} inicializaciones)\n" + "="*80)
print(f"\nSilhouette Score:")
print(f"  • Media: {np.mean(silhouette_scores_init):.4f}")
print(f"  • Desv. Est.: {np.std(silhouette_scores_init):.4f}")
print(f"  • Mín: {np.min(silhouette_scores_init):.4f}")
print(f"  • Máx: {np.max(silhouette_scores_init):.4f}")
print(f"  • Coef. Variación: {np.std(silhouette_scores_init)/np.mean(silhouette_scores_init):.4f}")

print(f"\nInercia:")
print(f"  • Media: {np.mean(inertias_init):.2f}")
print(f"  • Desv. Est.: {np.std(inertias_init):.2f}")
print(f"  • Mín: {np.min(inertias_init):.2f}")
print(f"  • Máx: {np.max(inertias_init):.2f}")

    # Visualizar distribución
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].hist(silhouette_scores_init, bins=15, color='steelblue', alpha=0.7, edgecolor='black')
axes[0].axvline(np.mean(silhouette_scores_init), color='red', linestyle='--', linewidth=2, label='Media')
axes[0].set_xlabel('Silhouette Score', fontweight='bold')
axes[0].set_ylabel('Frecuencia', fontweight='bold')
axes[0].set_title('Distribución de Silhouette Scores\n(50 inicializaciones diferentes)', fontsize=11, fontweight='bold')
axes[0].legend()
axes[0].grid(axis='y', alpha=0.3)

axes[1].hist(inertias_init, bins=15, color='coral', alpha=0.7, edgecolor='black')
axes[1].axvline(np.mean(inertias_init), color='red', linestyle='--', linewidth=2, label='Media')
axes[1].set_xlabel('Inercia', fontweight='bold')
axes[1].set_ylabel('Frecuencia', fontweight='bold')
axes[1].set_title('Distribución de Inertias\n(50 inicializaciones diferentes)', fontsize=11, fontweight='bold')
axes[1].legend()
axes[1].grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.show()

if np.std(silhouette_scores_init)/np.mean(silhouette_scores_init) < 0.05:
    print("\n✓ El algoritmo es muy estable (baja variabilidad en resultados)")
else:
    print("\n⚠ El algoritmo tiene cierta variabilidad (considerar más inicializaciones)")

## 10. Perfiles de Clusters

In [ ]:
# Crear perfiles descriptivos
print("\nPERFILES SUGERIDOS PARA CADA CLUSTER\n" + "="*80)

perfiles = {
    0: "Clientes Retail de Volumen Medio",
    1: "Clientes HoReCa Especializados",
    2: "Clientes Retail de Alto Volumen"
}

for cluster in range(k_optimo):
    df_cluster = df_original[df_original['K-means_3'] == cluster]
    
    # Identificar características dominantes
    gasto_promedio = df_cluster[variables_numericas].mean()
    categoria_mayor = gasto_promedio.idxmax()
    categoria_menor = gasto_promedio.idxmin()
    canal_prim = df_cluster['Channel'].mode()[0]
    
    canal_name = 'HoReCa' if canal_prim == 1 else 'Retail'
    gasto_total_prom = df_cluster[variables_numericas].sum(axis=1).mean()
    
    print(f"\n{'─'*80}")
    print(f"CLUSTER {cluster}: {perfiles[cluster]}")
    print(f"{'─'*80}")
    print(f"\n📊 Tamaño: {len(df_cluster)} clientes ({len(df_cluster)/len(df_original)*100:.1f}%)")
    print(f"💰 Gasto total promedio: ${gasto_total_prom:,.0f}")
    print(f"🏪 Canal predominante: {canal_name}")
    print(f"📈 Categoría con mayor gasto: {categoria_mayor}")
    print(f"📉 Categoría con menor gasto: {categoria_menor}")
    print(f"\n🎯 Estrategia Sugerida:")
    
    if cluster == 0:
        print("   • Enfoque en promociones de productos de larga duración")
        print("   • Optimizar distribución y logística")
        print("   • Programas de fidelización a largo plazo")
    elif cluster == 1:
        print("   • Productos frescos y perecederos premium")
        print("   • Servicio personalizado y entregas rápidas")
        print("   • Asociaciones estratégicas con cadenas gastronómicas")
    else:
        print("   • Volúmenes grandes con márgenes competitivos")
        print("   • Descuentos por volumen")
        print("   • Soluciones de suministro confiables y consistentes")

## 11. Resumen y Conclusiones

In [ ]:
resumen = f"""
{'='*80}
RESUMEN - K-MEANS CLUSTERING
{'='*80}

1. ALGORITMO APLICADO:
   ✓ K-means con k=3 clusters
   ✓ Inicialización: múltiples (n_init=10)
   ✓ Distancia: Euclidiana

2. RESULTADOS PRINCIPALES:
   • Inercia: {inertia_3:.2f}
   • Silhouette Score: {silhouette_3:.3f}
   • Davies-Bouldin Index: {davies_bouldin_3:.3f}
   • Calinski-Harabasz Score: {calinski_3:.2f}

3. DISTRIBUCIÓN DE CLUSTERS:
   • Cluster 0: {len(df_original[df_original['K-means_3']==0])} clientes ({len(df_original[df_original['K-means_3']==0])/len(df_original)*100:.1f}%)
   • Cluster 1: {len(df_original[df_original['K-means_3']==1])} clientes ({len(df_original[df_original['K-means_3']==1])/len(df_original)*100:.1f}%)
   • Cluster 2: {len(df_original[df_original['K-means_3']==2])} clientes ({len(df_original[df_original['K-means_3']==2])/len(df_original)*100:.1f}%)

4. COMPARACIÓN CON OTROS k:
   • k=3: Silhouette = {resultados_k[3]['silhouette']:.3f} ← RECOMENDADO
   • k=4: Silhouette = {resultados_k[4]['silhouette']:.3f}
   • k=5: Silhouette = {resultados_k[5]['silhouette']:.3f}

5. ESTABILIDAD:
   ✓ Algoritmo es estable con baja variabilidad entre inicializaciones
   ✓ Resultados reproducibles

6. VENTAJAS DEL MODELO:
   ✓ Interpretable y fácil de implementar
   ✓ Rápido computacionalmente
   ✓ Clara separación entre clusters
   ✓ Centroides informativos

7. PRÓXIMOS PASOS:
   → Comparar con Clustering Jerárquico
   → Evaluar DBSCAN
   → Seleccionar método final basado en interpretabilidad de negocio

{'='*80}
"""

print(resumen)

In [ ]:
print("✓ Análisis K-means completado exitosamente")
print("✓ Modelo guardado para comparación con otros métodos")